In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from run_metrics_periodic import run_metrics_periodic as run_metrics_original
from run_metrics_periodic_fixed import run_metrics_periodic as run_metrics_fixed

## Test Case 1: Object Wrapping Around Right Edge

Create a large circular object that wraps from the right edge to the left edge.

In [ ]:
def create_wrapped_circle(size=200, center_x=190, center_y=100, radius=30):
    """
    Create a circular object that wraps around the right boundary.
    """
    image = np.zeros((size, size), dtype=int)
    
    for i in range(size):
        for j in range(size):
            # Calculate distance with periodic boundaries
            dx = min(abs(j - center_x), size - abs(j - center_x))
            dy = min(abs(i - center_y), size - abs(i - center_y))
            dist = np.sqrt(dx**2 + dy**2)
            
            if dist <= radius:
                image[i, j] = 1
    
    return image

# Create test image
test_image_1 = create_wrapped_circle(size=200, center_x=190, center_y=100, radius=30)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(test_image_1, cmap='binary', origin='lower')
ax.set_title('Test Case 1: Object Wrapping Right→Left')
ax.set_xlabel('X (columns)')
ax.set_ylabel('Y (rows)')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Total pixels in object: {test_image_1.sum()}")

## Run Both Implementations on Test Case 1

In [ ]:
# Run original implementation
results_orig_1 = run_metrics_original(test_image_1.copy())
print("ORIGINAL Implementation:")
print(f"  Number of objects: {results_orig_1['number']}")
print(f"  Total area: {results_orig_1['area']}")
print(f"  Mean area: {results_orig_1['mean_area']:.1f}")

print("\nFIXED Implementation:")
# Run fixed implementation
results_fixed_1 = run_metrics_fixed(test_image_1.copy())
print(f"  Number of objects: {results_fixed_1['number']}")
print(f"  Total area: {results_fixed_1['area']}")
print(f"  Mean area: {results_fixed_1['mean_area']:.1f}")

print("\n" + "="*60)
print("EXPECTED RESULT:")
print("  Original should detect 2 objects (split by boundary)")
print("  Fixed should detect 1 object (properly connected)")
print("="*60)

## Test Case 2: Multiple Objects with One Wrapping

Create a scenario with multiple objects, where only one wraps around.

In [ ]:
def create_multiple_objects_with_wrapping(size=200):
    """
    Create multiple objects, with one wrapping around the boundary.
    """
    image = np.zeros((size, size), dtype=int)
    
    # Object 1: Normal object (interior)
    image[50:70, 50:70] = 1
    
    # Object 2: Normal object (interior)
    image[120:140, 100:120] = 1
    
    # Object 3: Wrapping object (crosses right boundary)
    image[150:170, 185:200] = 1  # Right part
    image[150:170, 0:5] = 1       # Left part (wrapped)
    
    # Object 4: Wrapping object (crosses bottom boundary)
    image[190:200, 80:100] = 1    # Bottom part
    image[0:10, 80:100] = 1        # Top part (wrapped)
    
    return image

test_image_2 = create_multiple_objects_with_wrapping(size=200)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(test_image_2, cmap='binary', origin='lower')
ax.set_title('Test Case 2: Multiple Objects with Boundary Wrapping')
ax.set_xlabel('X (columns)')
ax.set_ylabel('Y (rows)')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Run both implementations
results_orig_2 = run_metrics_original(test_image_2.copy())
results_fixed_2 = run_metrics_fixed(test_image_2.copy())

print("ORIGINAL Implementation:")
print(f"  Number of objects: {results_orig_2['number']}")
print(f"  Mean area: {results_orig_2['mean_area']:.1f}")

print("\nFIXED Implementation:")
print(f"  Number of objects: {results_fixed_2['number']}")
print(f"  Mean area: {results_fixed_2['mean_area']:.1f}")

print("\n" + "="*60)
print("EXPECTED RESULT:")
print("  Original: 6 objects (2 interior + 2×2 wrapped splits)")
print("  Fixed: 4 objects (2 interior + 2 wrapped properly connected)")
print("="*60)

## Analyze ACTUAL PINACLES Data for Boundary Crossing

Let's check if objects in your real data actually cross boundaries.

In [ ]:
import xarray as xr

# Load a sample from PINACLES
ds_2d1 = xr.open_dataset('/pscratch/sd/p/paccini/temp/temp_imse_budget/ds_2d_600x600_3km_SCREAMinit_part1.nc')
ds_2d_test = ds_2d1.isel(time=slice(0, 20))
model_2d_olr = ds_2d_test.toa_lw_up.copy()

def check_boundary_objects(image):
    """
    Check if any objects touch the domain boundaries.
    """
    # Check edges
    top_edge = image[0, :].sum()
    bottom_edge = image[-1, :].sum()
    left_edge = image[:, 0].sum()
    right_edge = image[:, -1].sum()
    
    return {
        'top': top_edge > 0,
        'bottom': bottom_edge > 0,
        'left': left_edge > 0,
        'right': right_edge > 0,
        'any': (top_edge + bottom_edge + left_edge + right_edge) > 0
    }

# Check several time steps
boundary_counts = {'top': 0, 'bottom': 0, 'left': 0, 'right': 0, 'any': 0}
n_timesteps = len(model_2d_olr.time)

for t in range(n_timesteps):
    ds_time = model_2d_olr.isel(time=t)
    image = np.where(ds_time.data < 173, 1, 0).astype(int)
    boundary_check = check_boundary_objects(image)
    
    for key in boundary_counts:
        if boundary_check[key]:
            boundary_counts[key] += 1

print("="*60)
print("BOUNDARY-TOUCHING OBJECTS IN PINACLES DATA")
print("="*60)
print(f"Time steps analyzed: {n_timesteps}")
print(f"\nTime steps with objects touching boundaries:")
print(f"  Top edge:    {boundary_counts['top']} ({boundary_counts['top']/n_timesteps*100:.1f}%)")
print(f"  Bottom edge: {boundary_counts['bottom']} ({boundary_counts['bottom']/n_timesteps*100:.1f}%)")
print(f"  Left edge:   {boundary_counts['left']} ({boundary_counts['left']/n_timesteps*100:.1f}%)")
print(f"  Right edge:  {boundary_counts['right']} ({boundary_counts['right']/n_timesteps*100:.1f}%)")
print(f"  Any edge:    {boundary_counts['any']} ({boundary_counts['any']/n_timesteps*100:.1f}%)")
print("="*60)

## Conclusion

**If synthetic tests show differences but PINACLES data doesn't:**
- Your original implementation is fine for this dataset
- Objects rarely wrap around boundaries in the 600x600 km domain
- The periodic boundary fix is unnecessary for your specific case

**If synthetic tests DON'T show differences:**
- The fix implementation might have issues
- Need to investigate the `stitch_periodic_boundaries()` function

**If PINACLES data shows many boundary-touching objects but no metric differences:**
- Objects may touch boundaries but don't actually wrap around
- Or they're small enough that splitting doesn't significantly affect metrics